# 04 · Baseline, iterate, freeze

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/04_prompt.ipynb)

Write the plainest prompt that could work, then improve it for reasons you can state.

```
  01_build_pool_<track>  →  02_sample  →  03_annotate  →▶ 04_prompt  →  05_report
```

| | |
|---|---|
| **Reads** | `data/gold/<track>_<group>_gold.json` (from 03) · the pool (from 01) |
| **Writes** | `outputs/<track>_<group>_predictions.json` · `..._rounds.json` |

---

Everything from here on is measured against **your** gold set, not the corpus's labels. That is the point of the last two notebooks.

> **Free-tier pacing.** The backend waits a few seconds between calls and retries on rate-limit errors, so a full run takes minutes and may print `(rate limited - waiting Ns then retrying)`. That is normal. Keep `N_PER_CLASS` small (2) while you iterate — then do **one** final run at full size.

## Setup — run this first

In Colab, uncomment **one** of the two clone blocks below before running. Colab starts with only this one file; the clone fetches everything *around* it (`scripts/`, `config.py`, `data/`) so the paths resolve.

**Do Option A once, as a group** — then always open the copy in Drive (*File ▸ Open ▸ Drive ▸ `lda2-final-template/notebooks/...`*). Your prompts, gold set and outputs then survive the runtime resetting, and everyone sees the same files.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first.
# ------------------------------------------------------------------
# In Google Colab, UNCOMMENT one of the two blocks below, then run the cell.

# --- Colab Option A: clone into your Google Drive (persists; do this once) ---
# from google.colab import drive
# drive.mount("/content/drive")
# %cd /content/drive/MyDrive
# ![ -d lda2-final-template ] || git clone https://github.com/egumasa/lda2-final-template.git lda2-final-template
# %cd /content/drive/MyDrive/lda2-final-template/notebooks

# --- Colab Option B: quick, throwaway clone (changes lost on reset) ---
# !git clone https://github.com/egumasa/lda2-final-template.git
# %cd lda2-final-template/notebooks

# Put scripts/ and config.py on the import path. Works locally AND in Colab
# after the %cd above, because notebooks/ sits beside both.
import sys
sys.path.append("../scripts")
sys.path.append("..")

from config import *      # TRACK, GROUP, SEED, N_PER_CLASS, and every path

from pipeline import *      # load_gold, load_prompt, run_prompt, save_json, ...
from metrics import *       # evaluate, agreement, show_errors

setup()                     # connect to the model and say which backend we got

describe()                  # what this notebook is working on


> **Check the backend line it just printed.** You want:
>
> ```
> LLM backend: Gemini API (gemini-3.1-flash-lite, temperature=0, seed=42)
> ```
>
> If it says **Colab Gemini** instead, no API key was found — put yours in the Colab Secrets panel (the 🔑 icon in the left sidebar) as `GEMINI_API_KEY` and re-run. The keyless backend has no temperature or seed, so the same prompt can give different answers and your numbers will not be reproducible. It must not be your final run.

> **Everything above comes from `config.py`** — one file at the top of the repo, which you edit once as a group. That is deliberate: the seed in notebook 02 has to be the seed in notebook 03, and five copies of a number in five notebooks is five chances for them to disagree. If the line it just printed is not your track, your group and your seed, fix `config.py` and re-run this cell.

In [ ]:
# ══ STEP 1 · Load your gold set and your pool ═════════════════════════════
# Goal      : the answers you score against, and the spare items few-shot draws from.
# Available : load_gold(path)  ->  a list of {id, text, label}
#             GOLD_PATH · POOL_PATH   (from config.py)
# Pointer   : Day 3 setup — the same call, twice.
# Produce   : gold · pool · LABELS      ← later cells use these names
# Note      : LABELS comes from your GOLD set, not the pool. If a label
#             never survived adjudication, it is not in your study.

# ✏️ your code here


## Step 2 — The baseline (round 0)

A number to beat. Write the plainest prompt that states the task and the label set, run it, score it. **Resist the urge to make it good** — the point of a baseline is that later rounds have something to be measured against, and a baseline you already tuned tells you nothing about whether tuning helped.

Your prompt lives in `prompts/<track>.txt` and must contain `{text}`, where each item gets slotted in. Edit the **file**, not a string in this notebook — that is what makes each version savable and comparable, and it is the reproducibility habit from S10.

In Colab you can write the file straight from a cell:

```python
%%writefile ../prompts/cefr_v0.txt
Classify the CEFR level of the sentence. Answer with the level only.
...

Sentence: {text}
```

In [ ]:
# ══ STEP 2 · Baseline prompt (round 0) ════════════════════════════════════
# Goal      : get one honest number to beat.
# Available : load_prompt(path)  ->  PROMPT
#             run_prompt(PROMPT, gold)  ->  predictions
#             evaluate(gold, predictions, ordered=..., labels=LABELS_ORDER)  ->  macro-F1
# Pointer   : Day 3 Part A — the same two lines.
# Produce   : f1_by_round · pred0      ← later cells use these names
# Note      : start f1_by_round = {} here and add a row per round. It is
#             the prompt-iteration table in your report, and notebook 05
#             reads it from a file.
# Careful   : keep N_PER_CLASS small for this. Full size is minutes of
#             pure waiting per round on the free tier.

# ✏️ your code here


## Step 3 — Iterate

Two or three more rounds. For each one, **change one thing, predict what it will do, then find out.** "Added examples" is not a reason; "the model kept confusing B1 and B2, so I gave it one example of each" is. Write the reason down as you go — reconstructing it afterwards from a stack of F1 numbers is much harder than it sounds, and it is report section 2.

A round that made things **worse** is a result, not a mistake. Keep it in the table. It is often the most informative row you have.

`build_fewshot` draws examples from the pool while avoiding anything in your gold set — otherwise you would be testing the model on answers you had just shown it.

In [ ]:
# ══ STEP 3 · Iterate — 2–3 reasoned rounds ════════════════════════════════
# Goal      : improve the prompt for stated reasons, and record what each change did.
# Available : build_fewshot(PROMPT, pool, gold)  ->  a new prompt with examples
#             run_prompt(...)  ·  evaluate(...)   (as in step 2)
# Pointer   : Day 3 iterations 1–2. build_fewshot replaces typing the examples by hand.
# Produce   : PROMPT (your best one) · f1_by_round      ← later cells use these names
# Note      : save each version as its own prompt file (v0, v1, v2). A
#             prompt you overwrote is a round you cannot report.
# Ask       : did the confusion matrix change SHAPE, or did everything
#             shift a little? Those need different next moves.

# ✏️ your code here


## Step 4 — Freeze

A hosted model is only *best-effort* reproducible, even at `temperature=0`. So once your best prompt is settled:

1. Raise `N_PER_CLASS` in `config.py` to full size — and re-run notebooks 02 and 03 if that changes your sample. (If it does, you have more annotating to do. This is why you decide the size **before** you annotate.)
2. Run the model **once**, on your best prompt.
3. `save_json` the predictions to a file.

Every number you report from here on comes out of that file. That is what makes your F1 hold still, and what lets anyone else re-run your analysis on exactly the outputs you saw.

**One person runs this.** It is the run you will be defending.

In [ ]:
# ══ STEP 4 · The final run, frozen to a file ══════════════════════════════
# Goal      : one full-size run on your best prompt, saved so the numbers stop moving.
# Available : run_prompt(PROMPT, gold)  ->  predictions
#             save_predictions(predictions, PRED_PATH)  ·  load_predictions(PRED_PATH)
#             save_json(f1_by_round, ROUNDS_PATH, what="rounds")
# Pointer   : Day 2 S6 loaded a frozen file we made; now you make your own.
# Produce   : pred_final      ← later cells use these names
# Freeze    : save, then load it straight back and use THAT from now on.
#             Reading it back is not superstition — it is the check that
#             the file you will report from is the file you think it is.
# Careful   : save f1_by_round too. Notebook 05 needs it, and it is the
#             one thing here that exists only in this session's memory.

# ✏️ your code here


---

**Next:** open `05_report.ipynb`. It loads the two files you just wrote and nothing else — so from here on, your numbers cannot move.